In [1]:
import sys
if '/disks/cosmodm/vdvuurst' not in sys.path:
    sys.path.append('/disks/cosmodm/vdvuurst')

import numpy as np
import h5py
from matplotlib import pyplot as plt
import os
from importlib import reload
import json
import ONEHALO
from tqdm import tqdm
from functions import modified_logspace, BIC
from onehalo_plotter import *
from functional_forms import *
from scipy.optimize import minimize, brentq

format_plot()


In [16]:
test = np.arange(10)
idx = 9
x = 11
np.hstack([test[:idx], x, test[idx+1:]])

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 11])

In [ ]:

### FOR PERTURBING AROUND THE OPTIMUM FROM MCMC
def sample_for_brent_ND(params, MCMC_stepsizes, found_likelihood, log_likelihood_func, single_gauss = False, threshold = 1.01):
    # if single_gauss:
    #     step_sizes = [100.]
    # else:
    #     step_sizes = [100., 100., 0.1] # high valued steps relative to global prior

    step_sizes = MCMC_stepsizes * 100 # indicative of the parameter value range, but larger steps for speed

    Lthresh = threshold * found_likelihood # desired likelihood decrease, 1% by default

    bounds = [[] for _ in range(len(step_sizes))]

    for param_idx,step in tqdm(enumerate(step_sizes)):
        param_insert = params.copy()
        current_param_value = params[param_idx]

        # step in the positive direction
        # prior_flag = False
        Lcurrent = found_likelihood
        n = 0
        while Lcurrent < Lthresh:
            current_param_value += step
            # if current_param_value >= GLOBAL_PRIOR_RANGE[param_idx][1]:
            #     prior_flag = True
            #     break

            #inplace operation on param_insert to change the relevant parameter
            np.put(param_insert, param_idx, current_param_value)
            Lcurrent = log_likelihood_func(param_insert)
            n += 1
            if n == 100:
                break

        param_up = current_param_value #if not prior_flag else GLOBAL_PRIOR_RANGE[param_idx][1]

        # step down, so reset some stuff
        # prior_flag = False
        np.put(param_insert, param_idx, params[param_idx])
        current_param_value = params[param_idx]
        Lcurrent = found_likelihood
        n = 0
        while Lcurrent < Lthresh:
            current_param_value -= step
            # if current_param_value <= GLOBAL_PRIOR_RANGE[param_idx][0]:
            #     prior_flag = True
            #     break

            #inplace operation on param_insert to change the relevant parameter
            np.put(param_insert, param_idx, current_param_value)
            Lcurrent = log_likelihood_func(param_insert)
            n += 1
            if n == 100:
                break

        param_down = current_param_value #if not prior_flag else GLOBAL_PRIOR_RANGE[param_idx][0]

        bounds[param_idx] = [param_down, param_up]
    
    return bounds

def _generate_perturb_func(x, params, x_idx, log_likelihood_func, L_thresh):
    return (log_likelihood_func(
        np.hstack([params[:x_idx], x, params[x_idx+1:]]) - (L_thresh)
        )
    )

def _perturb_params_ND(params: np.array, log_likelihood_func: Callable[[np.array], float], MCMC_stepsizes: np.ndarray,
                        single_gauss: bool = False, threshold: float = 1.01):
    """ Perturb parameter value, either upwards or downwards, and return parameter value when likelihood threshold is reached.

    Args:
        params (list or array): list or array containing the parameter values *in order* [sigma1, sigma2, lambda]
        L_thresh (float): Threshold likelihood to reach.
        log_likelihood_func (function, optional): Log likelihood of the model. Defaults to double_gaussian_log_likelihood. NOTE that this is an L-maximizing function since it is negative numbers we want to be as near to 0 as possible!
        single_gaussian (bool, optional): Controls whether we are fitting a single gaussian model (True) or a Double Gaussian (False). Defaults to False.
        threshold (float, optional): How much we want to be above the best found likelihood (relative). Defaults to 1.01 (1%). 
    
    Returns:
        param_bounds (list): nested list of 2-lists holding found parameter bounds at which the likelihood meets the set threshold (lower, upper)
        opt_params (np.ndarray): array holding results of scipy.optimize.minimize to find the local optimum near the input values. 
    """

    opt_params = minimize(log_likelihood_func, x0 = params).x
    L_best = log_likelihood_func(opt_params)

    # function which perturb only 1 parameter at a time, need to do it like this because the argument position changes
    L_thresh = threshold * L_best
    perturb_funcs = [lambda x: _generate_perturb_func(x, opt_params, i, log_likelihood_func, L_thresh) for i in range(len(params))]

    # get the bounds for the root finder by taking large samples. note that this is actually the bottleneck of the code
    brent_bounds = sample_for_brent_ND(opt_params, MCMC_stepsizes, L_best, log_likelihood_func, single_gauss = single_gauss, threshold=threshold)

    param_bounds = [[] for _ in range(len(params))]

    # NOTE runtime error means it did not converge, value error means that the signage of the function 
    # didnt change in the interval so there was no root in it    
    if not single_gauss:
        for param_idx in range(len(params)):
            try:
                lower_bound = brentq(perturb_funcs[param_idx], brent_bounds[param_idx][0], opt_params[param_idx])
            except (RuntimeError, ValueError):
                lower_bound = None
            try:
                upper_bound = brentq(perturb_funcs[param_idx], opt_params[param_idx], brent_bounds[param_idx][1])
            except (RuntimeError, ValueError):
                upper_bound = None
            
            param_bounds[param_idx] = [lower_bound, upper_bound]

    #TODO: single_gauss will not work for joint models, do we want it to?
    else:
        # sigma_1 needs this root-finding optimization
        try:
            lower_bound = brentq(perturb_funcs[0], brent_bounds[0][0], opt_params[0])
        except (RuntimeError, ValueError):
            lower_bound = GLOBAL_PRIOR_RANGE[0][0]
        try:
            upper_bound = brentq(perturb_funcs[0], opt_params[0], brent_bounds[0][1])
        except (RuntimeError, ValueError):
            upper_bound = GLOBAL_PRIOR_RANGE[0][1]
        
        param_bounds[0] = [lower_bound, upper_bound]
        
        # the other two parameters do not need it and just get the prior range as errors
        for param_idx in range(1,len(params)):
            param_bounds[param_idx] = GLOBAL_PRIOR_RANGE[param_idx]

    return param_bounds, opt_params

def perturb_around_likelihood_ND(params: np.ndarray, log_likelihood_func: Callable[[np.ndarray], float], MCMC_stepsizes: np.ndarray,
                                 single_gauss: bool = False, threshold: float = 1.01) -> dict:
    """ Given a maximum from MCMC, perturb around this found maximum in small steps to estimate the error and refine the estimate.
        Uses the _perturb_params() functionality as a workhorse, this is moreso a wrapper around that function to format it to a dictionary

    Args:
        params (np.array): Best found parameter set
        log_likelihood_func (function, optional): Log likelihood of the model. Defaults to double_gaussian_log_likelihood.
        single_gauss (bool, optional): Controls whether we "turn off" the lambda parameter, i.e. fix it to 0.
        threshold (float, optional): How much we want to be above the best found likelihood (relative). Defaults to 1.01 (1%). 

    Returns:
        param_dict: dictionary of parameter values and error estimates
    """
    
    #TODO: fix for joint model, there will be more parameters and this is hardcoded to 3 parameters now
    # ^that is gonna suck so fucking much

    bounds, best_params = _perturb_params_ND(params, log_likelihood_func, MCMC_stepsizes, single_gauss = single_gauss, threshold = threshold)
    param_dict =  {}
    param_dict['params'] = best_params
    param_dict['errors'] = [[] for _ in range(len(best_params))]

    for i in range(len(best_params)):
        if np.any(np.array(bounds) == None):
            param_dict['errors'][i] = [None, None]
        else: 
            param_down, param_up = bounds[i]
            param_dict['errors'][i] = [np.abs(best_params[i] - param_down), np.abs(best_params[i] - param_up)] # errors are the parameter differences

    return param_dict

In [53]:
test_result_path = '/disks/cosmodm/vdvuurst/data/OneHalo_param_fits/joint_subsample/Nelder-Mead/function_combi_2000.json'
with open(test_result_path) as f:
    test_dict = json.load(f)

init_params, MCMC_stepsizes = np.load('/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_2000.npy')

function_combi = all_combis[2000 - 1]
params = np.array(test_dict['parameters'])
L = test_dict['likelihood']
n_params_r, n_params_m, ntot = ONEHALO.param_info(function_combi)
joint_fitter = ONEHALO.ONEHALO_joint_fitter()

ll_func = lambda x: joint_fitter.get_joint_likelihood(x, n_params_m, n_params_r, function_combi)

refined_param_dict = perturb_around_likelihood_ND(params, ll_func, MCMC_stepsizes, threshold = 1.0001)
refined_param_dict

8it [00:00, 75.65it/s]/disks/cosmodm/vdvuurst/ONEHALO.py:700: RuntimeWarning: overflow encountered in cast
  
24it [00:14,  1.34it/s]/disks/cosmodm/vdvuurst/functional_forms.py:15: RuntimeWarning: overflow encountered in exp
  return A * np.exp(-B * x) + C
26it [00:16,  1.59it/s]


{'params': array([ 8.17437255e+01, -4.73690335e+00, -3.43971366e+00, -1.90819028e+02,
        -5.10230577e+01,  4.51085478e+01,  5.01657182e-01,  2.34748393e+00,
        -3.69920839e+01,  2.99941166e-01,  2.92419947e+02,  1.14016844e+00,
        -3.81979591e+00,  1.46654624e+01,  1.64244831e-01,  9.31550554e+00,
        -5.24026124e-01,  1.14305522e+00, -1.94881518e-01, -1.17157772e+05,
         1.86927857e+00,  4.06459080e+00, -1.71395791e+00, -3.16812839e-01,
         1.29449999e+01, -4.50613461e-01]),
 'errors': [[None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None],
  [None, None]]}

In [54]:
refined_L = ll_func(refined_param_dict['params'])
print(refined_param_dict['params'], 'L:', refined_L)
print(params, 'L:', L)
print('refined better:', refined_L < L)

[ 8.17437255e+01 -4.73690335e+00 -3.43971366e+00 -1.90819028e+02
 -5.10230577e+01  4.51085478e+01  5.01657182e-01  2.34748393e+00
 -3.69920839e+01  2.99941166e-01  2.92419947e+02  1.14016844e+00
 -3.81979591e+00  1.46654624e+01  1.64244831e-01  9.31550554e+00
 -5.24026124e-01  1.14305522e+00 -1.94881518e-01 -1.17157772e+05
  1.86927857e+00  4.06459080e+00 -1.71395791e+00 -3.16812839e-01
  1.29449999e+01 -4.50613461e-01] L: 251729.6132644054
[ 8.17411327e+01 -4.73780107e+00 -3.43613522e+00 -1.90820822e+02
 -5.10237233e+01  4.51152725e+01  5.04376783e-01  2.34763332e+00
 -3.69916188e+01  3.56615604e-01  2.92423255e+02  1.14016844e+00
 -3.81979591e+00  1.46654624e+01  1.64244831e-01  9.31550554e+00
 -5.24026124e-01  1.14305522e+00 -1.94881518e-01 -1.17157772e+05
  1.86927857e+00  4.06459080e+00 -1.71395791e+00 -3.16812839e-01
  1.29449999e+01 -4.50613461e-01] L: 251732.0331629306
refined better: True


In [55]:
params

array([ 8.17411327e+01, -4.73780107e+00, -3.43613522e+00, -1.90820822e+02,
       -5.10237233e+01,  4.51152725e+01,  5.04376783e-01,  2.34763332e+00,
       -3.69916188e+01,  3.56615604e-01,  2.92423255e+02,  1.14016844e+00,
       -3.81979591e+00,  1.46654624e+01,  1.64244831e-01,  9.31550554e+00,
       -5.24026124e-01,  1.14305522e+00, -1.94881518e-01, -1.17157772e+05,
        1.86927857e+00,  4.06459080e+00, -1.71395791e+00, -3.16812839e-01,
        1.29449999e+01, -4.50613461e-01])